# 📄 RAG-Based PDF Chatbot

This notebook builds a Retrieval-Augmented Generation (RAG) system to answer questions from a PDF.

Pipeline:
PDF → Chunking → Embeddings → FAISS → Retrieval → LLM → Answer

In [ ]:
!pip install -U langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers transformers pypdf

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print('Uploaded:', file_name)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(file_name)
documents = loader.load()
print('Total Pages:', len(documents))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.split_documents(documents)
print('Total Chunks:', len(docs))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')
db = FAISS.from_documents(docs, embeddings)
print('Vector DB Created')

In [ ]:
from transformers import pipeline
generator = pipeline('text-generation', model='tiiuae/falcon-rw-1b')

In [ ]:
def ask_question(query):
    results = db.similarity_search(query, k=8)
    context = ' '.join([doc.page_content for doc in results])

    prompt = f"""
You are a machine learning expert.
Answer clearly in 2-3 sentences using ONLY the context.

Context:
{context}

Question:
{query}

Answer:
"""

    response = generator(prompt, max_new_tokens=120, do_sample=True, temperature=0.3)
    output = response[0]['generated_text']
    answer = output.split('Answer:')[-1].strip()

    print('Question:', query)
    print('Answer:', answer)

In [ ]:
ask_question('What is bias variance tradeoff?')
ask_question('What is RAG?')
ask_question('Explain vector database')